# Run Star
- Just make a function that runs the whole star, returning the predicted best in x and predictions.
- Have an option to plot.
- There is no test set here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import emcee
from scipy.optimize import minimize
from scipy.stats import norm
import celerite2
from celerite2 import terms
from prettytable import PrettyTable
import sys
from pathlib import Path
from IPython.display import display as ipy_display

sys.path.append(str(Path('../..').resolve()))
from helpers.df_ops import prepare_df, split_df, clean_df, downsample_min_gap
from helpers.priors import find_classify_signals, get_priors
from helpers.gpr import set_params, NLL
from helpers.MCMC import lnPost_gp
from helpers.eval import check_constant, best_in_x, fit_Fourier, truth_in_x

In [ ]:
def run_star(datapath, star_name, star_type='G', add_prefix=False,
             error_percent=2.5, sigma_upper_mult=5.0, q_bounds_in=(1, 5),
             min_gap=3, n_walkers=32, total_sample_count=2500, subsample=500, SEED=1701,
             pred_forward_years=2, high_cad=10,
             direct_bound_tol=0.1, sm2016_bound_tol=0.2, mean_bound_tol=1,
             require_mid=True, valid_metric='CRPS', lookahead_years=3,
             verbose=False, plot=False):
    '''
    Run the full GPR on one star using a 75/25 train/validation split.
    No test set. Returns predictions and predicted best-in-x.

    Returns dict:
        preds          - (subsample, n_times) array of MCMC prediction draws
        sampled_years  - (n_times,) year grid for predictions
        best_med_year  - median predicted minimum year within lookahead_years
        best_lb_year   - 16th percentile
        best_ub_year   - 84th percentile
        best_combo     - selected kernel combination name
    '''
    np.random.seed(SEED)

    #---Data Loading---
    raw_df = pd.read_csv(datapath, sep=r'\s+', skip_blank_lines=True)
    data_df = prepare_df(raw_df, add_prefix=add_prefix, relative=True)
    data_df = downsample_min_gap(data_df, min_gap)

    #---Split 75/25---
    dirty_train_df, dirty_valid_df, _ = split_df(data_df, train_split=0.75, valid_split=0.25)
    train_df, valid_df, MAD = clean_df(dirty_train_df, dirty_valid_df, tol=4, verbose=verbose, plot=False)

    #---Priors---
    classified_signal_data = find_classify_signals(
        dirty_train_df,
        plot_fitpeaks=False, verbose_fitpeaks=False,
        plot_genpriors=False, verbose_genpriors=False)
    rho_priors, rho_prior_bounds, _ = get_priors(
        classified_signal_data, star_type=star_type,
        direct_bound_tol=direct_bound_tol, sm2016_bound_tol=sm2016_bound_tol,
        mean_bound_tol=mean_bound_tol, verbose=verbose)

    #---Model Selection---
    prior_combos = {
        '1m':   {'k': 1, 'q_priors': None, 'cycle_keys': ['mid']},
        '2sm':  {'k': 2, 'q_priors': None, 'cycle_keys': ['short', 'mid']},
        '2ml':  {'k': 2, 'q_priors': None, 'cycle_keys': ['mid', 'long']},
        '3sml': {'k': 3, 'q_priors': None, 'cycle_keys': ['short', 'mid', 'long']},
    }
    if require_mid:
        prior_combos = {nm: v for nm, v in prior_combos.items() if 'mid' in v['cycle_keys']}

    train_yerr = train_df['sind'] * error_percent / 100
    train_mean = train_df['sind'].mean()
    train_std  = train_df['sind'].std()

    combo_results = {}
    for combo_name, combo_info in prior_combos.items():
        k_c = combo_info['k']
        q_prior_type = combo_info['q_priors']
        cycle_keys_c = combo_info['cycle_keys']

        np.random.seed(SEED)
        q_0s     = ([np.random.uniform(0, 0.5) for _ in range(k_c)]
                    if q_prior_type == 'overdamped'
                    else [np.random.uniform(0.5, 1) for _ in range(k_c)])
        sigma_0s = [train_std / k_c for _ in range(k_c)]
        rho_0s   = [rho_priors[ck] for ck in cycle_keys_c]

        ig_raw   = np.concatenate([sigma_0s, rho_0s, q_0s])
        q_bnd    = ([(- np.inf, np.log(0.5)) for _ in range(k_c)]
                    if q_prior_type == 'overdamped'
                    else [(np.log(q_bounds_in[0]), np.log(q_bounds_in[1])) for _ in range(k_c)])
        sigma_bnd = [(np.log(1e-4), np.log(train_std * sigma_upper_mult)) for _ in range(k_c)]
        rho_bnd   = np.log([rho_prior_bounds[ck] for ck in cycle_keys_c])
        bounds_c  = np.concatenate([sigma_bnd, rho_bnd, q_bnd])

        perturbs  = ig_raw * 0.1 * np.random.normal(size=(25, len(ig_raw)))
        perturbed = np.log(np.clip(ig_raw + perturbs, 1e-6, None))

        best_NLL, best_res_c = np.inf, None
        for ig in perturbed:
            kernel = terms.SHOTerm(sigma=sigma_0s[0], rho=rho_0s[0], Q=q_0s[0])
            for ki in range(1, k_c):
                kernel += terms.SHOTerm(sigma=sigma_0s[ki], rho=rho_0s[ki], Q=q_0s[ki])
            gp = celerite2.GaussianProcess(kernel, mean=train_mean)
            gp.compute(train_df['day'], yerr=train_yerr)
            res = minimize(NLL, ig, args=(gp, k_c, train_df['sind'].to_numpy()),
                           method='L-BFGS-B', bounds=bounds_c)
            if res.fun < best_NLL:
                best_NLL, best_res_c = res.fun, res

        gp = set_params(best_res_c.x, k_c, gp)
        gp.recompute()
        bp = np.exp(best_res_c.x)

        mu, cov     = gp.predict(train_df['sind'], t=valid_df['day'], return_var=True)
        y_valid     = valid_df['sind'].to_numpy()
        valid_yerr  = valid_df['sind'] * error_percent / 100
        total_var   = cov + valid_yerr ** 2
        total_std   = np.sqrt(total_var)

        nlpd = (0.5 * np.log(2 * np.pi * total_var) + (y_valid - mu) ** 2 / (2 * total_var)).mean()
        z    = (y_valid - mu) / total_std
        crps = (total_std * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))).mean()

        combo_results[combo_name] = {'NLPD': nlpd, 'CRPS': crps, 'params': bp, 'bounds': bounds_c}

    best_metric, best_combo_name = np.inf, None
    for cname, cr in combo_results.items():
        if cr[valid_metric] < best_metric:
            best_metric, best_combo_name = cr[valid_metric], cname
            best_params = cr['params']
            best_bounds = cr['bounds']

    if verbose:
        print(f"{star_name}: selected {best_combo_name} ({valid_metric}={best_metric:.4f})")

    #---Retrain on all data---
    retrain_df   = downsample_min_gap(pd.concat([train_df, valid_df]), min_gap)
    k            = prior_combos[best_combo_name]['k']
    retrain_yerr = retrain_df['sind'] * error_percent / 100
    retrain_mean = retrain_df['sind'].mean()

    p0     = best_params
    kernel = terms.SHOTerm(sigma=p0[0], rho=p0[k], Q=p0[2*k])
    for ki in range(1, k):
        kernel += terms.SHOTerm(sigma=p0[ki], rho=p0[k + ki], Q=p0[2*k + ki])

    gp_retrain = celerite2.GaussianProcess(kernel, mean=retrain_mean)
    gp_retrain.compute(retrain_df['day'], yerr=retrain_yerr)

    retrain_res = minimize(NLL, np.log(best_params),
                           args=(gp_retrain, k, retrain_df['sind'].to_numpy()),
                           method='L-BFGS-B', bounds=best_bounds)
    gp_retrain    = set_params(retrain_res.x, k, gp_retrain)
    gp_retrain.recompute()
    retrain_params = np.exp(retrain_res.x)

    #---MCMC Posterior---
    np.random.seed(SEED)
    y   = retrain_df['sind'].to_numpy()
    wsc = np.log(retrain_params) + 1e-5 * np.random.randn(n_walkers, len(retrain_params))
    sampler = emcee.EnsembleSampler(
        n_walkers, len(retrain_params), lnPost_gp,
        args=(gp_retrain, k, y, retrain_res.x, best_bounds))
    sampler.run_mcmc(wsc, nsteps=total_sample_count, progress=False)

    ln_chains  = sampler.get_chain(discard=1000)
    ln_samples = ln_chains.reshape(-1, ln_chains.shape[-1])
    np.random.seed(SEED)
    sel        = np.random.choice(len(ln_samples), size=subsample, replace=False)
    ln_samples = ln_samples[sel]

    #---Forecast Grid---
    t0_day            = retrain_df['day'].iloc[0]
    t0_year           = retrain_df['year'].iloc[0]
    t_pred_start      = retrain_df['day'].iloc[-1]
    t_pred_start_year = retrain_df['year'].iloc[-1]
    sampled_days      = np.arange(t_pred_start, t_pred_start + 365 * pred_forward_years, high_cad)
    sampled_years     = t0_year + (sampled_days - t0_day) / 365.25

    preds = []
    for ln_s in ln_samples:
        set_params(ln_s, k, gp_retrain)
        gp_retrain.recompute(quiet=True)
        p, _ = gp_retrain.predict(retrain_df['sind'], t=sampled_days, return_var=True)
        preds.append(p)
    preds = np.array(preds)

    #---Best in X---
    best_med, (best_lb, best_ub) = best_in_x(preds, sampled_days, lookahead_years, t_start=t_pred_start)
    to_year = lambda d: t0_year + (d - t0_day) / 365.25
    best_med_year = to_year(best_med)
    best_lb_year  = to_year(best_lb)
    best_ub_year  = to_year(best_ub)

    #---Plot---
    if plot:
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.scatter(retrain_df['year'], retrain_df['sind'], s=5, color='steelblue', label='Data', zorder=3)
        mean_pred = preds.mean(axis=0)
        lb68      = np.percentile(preds, 16, axis=0)
        ub68      = np.percentile(preds, 84, axis=0)
        ax.plot(sampled_years, mean_pred, color='C1', label='GPR mean')
        ax.fill_between(sampled_years, lb68, ub68, color='C1', alpha=0.3, label='68%')
        ax.axvline(best_med_year, color='red', lw=1.2,
                   label=f'Best in {lookahead_years}yr: {best_med_year:.2f}')
        ax.axvspan(best_lb_year, best_ub_year, color='red', alpha=0.12)
        ax.set_xlabel('Year')
        ax.set_ylabel('S-index')
        ax.set_title(star_name)
        ax.legend()
        plt.tight_layout()
        plt.show()

    return {
        'preds':         preds,
        'sampled_years': sampled_years,
        'best_med_year': best_med_year,
        'best_lb_year':  best_lb_year,
        'best_ub_year':  best_ub_year,
        'best_combo':    best_combo_name,
    }